In [ ]:

import os
import time
import requests
import pandas as pd
from google.colab import files

# ==========================================
# CONFIGURAÇÕES INICIAIS
# ==========================================

ARQUIVO_PLANILHA = "caminhoes.xlsx"
ARQUIVO_RESULTADO = "resultado_fipe.xlsx"

COL_MODELO = "MODELO"
COL_FABRICANTE = "FABRICANTE"
COL_ANO = "ANO MODELO"
COL_CODIGO_FIPE = "CODIGO FIPE"
COL_VALOR_FIPE = "TABELA FIPE"

BASE_URL = "https://fipe.parallelum.com.br/api/v2"
HEADERS = {"User-Agent": "Mozilla/5.0"}
TIPO_SLUG = "trucks"

# ==========================================
# FUNÇÃO DE REQUISIÇÃO COM RETRY
# ==========================================

def get_com_retry(url, tentativas=3, espera=2):
    """
    Realiza uma requisição HTTP com número limitado de tentativas.
    Em caso de erro de conexão ou erro do servidor, aguarda alguns
    segundos antes de tentar novamente.
    """
    for tentativa in range(tentativas):
        try:
            res = requests.get(
                url,
                headers=HEADERS,
                timeout=20
            )

            if res.status_code == 200:
                return res

            if res.status_code >= 500 and tentativa < tentativas - 1:
                print(
                    f"Status {res.status_code}. "
                    f"Tentativa {tentativa + 1}/{tentativas}."
                )
                time.sleep(espera * (tentativa + 1))
                continue

            return res

        except requests.exceptions.RequestException as e:
            if tentativa < tentativas - 1:
                print(
                    f"Erro de conexão. "
                    f"Tentativa {tentativa + 1}/{tentativas}."
                )
                time.sleep(espera * (tentativa + 1))
                continue

            print(
                f"Falha após {tentativas} tentativas: {e}"
            )
            return None

    return None


# ==========================================
# BUSCA POR CÓDIGO FIPE
# ==========================================

def buscar_por_codigo_fipe(codigo_fipe, ano_alvo):
    """
    Consulta a API da FIPE utilizando o código FIPE informado.

    Primeiro verifica os anos disponíveis para o código FIPE.
    Em seguida, identifica o ano correspondente e consulta
    os dados e o preço do veículo.

    Retorna:
        dados_preco: dados do veículo encontrados na API
        status: situação da consulta
        anos_disponiveis: anos disponíveis para o código FIPE
    """

    codigo_fipe = str(codigo_fipe).strip()

    res_anos = get_com_retry(
        f"{BASE_URL}/{TIPO_SLUG}/{codigo_fipe}/years"
    )

    if not res_anos or res_anos.status_code != 200:
        return None, "Código FIPE inválido ou não encontrado", ""

    anos = res_anos.json()

    if not anos:
        return None, "Nenhum ano disponível para esse código FIPE", ""

    anos_disponiveis_str = ", ".join(
        sorted({a["name"] for a in anos})
    )

    id_ano_exato = None

    for a in anos:
        if a["code"].startswith(str(ano_alvo)):
            id_ano_exato = a["code"]
            break

    if not id_ano_exato:
        return (
            None,
            f"Ano {ano_alvo} não existe para este código FIPE",
            anos_disponiveis_str
        )

    res_preco = get_com_retry(
        f"{BASE_URL}/{TIPO_SLUG}/{codigo_fipe}/years/{id_ano_exato}"
    )

    if not res_preco or res_preco.status_code != 200:
        return None, "Erro ao buscar preço", anos_disponiveis_str

    return res_preco.json(), "OK", anos_disponiveis_str


# ==========================================
# PROCESSAMENTO DA PLANILHA
# ==========================================

if not os.path.exists(ARQUIVO_PLANILHA):

    print(
        f"Erro: o arquivo '{ARQUIVO_PLANILHA}' "
        "não foi encontrado no ambiente do Colab."
    )

else:

    print(f"Carregando a planilha '{ARQUIVO_PLANILHA}'...")

    df = pd.read_excel(ARQUIVO_PLANILHA)

    # Cria a coluna de valor FIPE caso ela ainda não exista
    if COL_VALOR_FIPE not in df.columns:
        df[COL_VALOR_FIPE] = ""

    # Criação das colunas de controle e informações da consulta
    df["FIPE_Status"] = ""
    df["FIPE_Nome_Oficial"] = ""
    df["FIPE_Mes_Referencia"] = ""
    df["FIPE_Anos_Disponiveis"] = ""

    print(f"Total de registros: {len(df)}\n")

    # Processamento dos registros da planilha
    for index, row in df.iterrows():

        modelo = str(
            row.get(COL_MODELO, "")
        ).strip()

        fabricante = str(
            row.get(COL_FABRICANTE, "")
        ).strip()

        ano_modelo = str(
            row.get(COL_ANO, "")
        ).strip()

        codigo_fipe = str(
            row.get(COL_CODIGO_FIPE, "")
        ).strip()

        # Remove casas decimais do ano quando o Excel
        # interpreta o valor como número decimal
        if "." in ano_modelo:
            ano_modelo = ano_modelo.split(".")[0]

        # Remove ".0" quando o código FIPE é interpretado
        # como número pelo Excel
        if codigo_fipe.endswith(".0"):
            codigo_fipe = codigo_fipe[:-2]

        # Verifica se existe código FIPE
        if not codigo_fipe or codigo_fipe.lower() == "nan":

            df.at[
                index,
                "FIPE_Status"
            ] = "Sem código FIPE informado"

            continue

        print(
            f"[{index + 1}/{len(df)}] "
            f"{fabricante} {modelo} ({ano_modelo}) | "
            f"Código FIPE: {codigo_fipe}"
        )

        dados, status, anos_disp = buscar_por_codigo_fipe(
            codigo_fipe,
            ano_modelo
        )

        df.at[
            index,
            "FIPE_Anos_Disponiveis"
        ] = anos_disp

        if dados:

            print(
                f"Valor FIPE: {dados['price']}"
            )

            df.at[
                index,
                COL_VALOR_FIPE
            ] = dados["price"]

            df.at[
                index,
                "FIPE_Status"
            ] = "Sucesso"

            df.at[
                index,
                "FIPE_Nome_Oficial"
            ] = dados["model"]

            df.at[
                index,
                "FIPE_Mes_Referencia"
            ] = dados["referenceMonth"]

        else:

            print(
                f"Consulta não realizada: {status}"
            )

            df.at[
                index,
                "FIPE_Status"
            ] = status

        # Intervalo entre as requisições
        time.sleep(1.0)

    # ==========================================
    # RESUMO E EXPORTAÇÃO
    # ==========================================

    print("\nResumo das consultas:")
    print(df["FIPE_Status"].value_counts())

    df.to_excel(
        ARQUIVO_RESULTADO,
        index=False
    )

    print(
        f"\nProcessamento concluído. "
        f"Arquivo gerado: '{ARQUIVO_RESULTADO}'"
    )

    files.download(ARQUIVO_RESULTADO)

📦 Carregando a planilha 'caminhoes.xlsx'...
Total de registros: 42

[1/42] MERCEDES-BENZ ATEGO 2426 CL (2022) | código 509284-1...
   ↳ ✅ R$ 477.302,00


/tmp/ipykernel_1448/3717655744.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'R$ 477.302,00' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index, COL_VALOR_FIPE] = dados['price']


[2/42] MERCEDES-BENZ ACCELO 1016 2p (2022) | código 509279-5...
   ↳ ✅ R$ 297.841,00
[3/42] MERCEDES-BENZ 1214 (1997) | código 509002-4...
   ↳ ✅ R$ 57.546,00
[4/42] MERCEDES-BENZ 710 (1999) | código 509032-6...
   ↳ ✅ R$ 93.752,00
[5/42] MERCEDES-BENZ 1214 (1996) | código 509002-4...
   ↳ ✅ R$ 52.108,00
[6/42] VOLKSWAGEN 8150 (2001) | código 515065-5...
   ↳ ✅ R$ 119.795,00
[7/42] MERCEDES-BENZ 710 (2009) | código 509032-6...
   ↳ ✅ R$ 145.372,00
[8/42] MERCEDES-BENZ 1214 (1995) | código 509002-4...
   ↳ ✅ R$ 46.789,00
[9/42] MERCEDES-BENZ 710 (2000) | código 509032-6...
   ↳ ✅ R$ 100.211,00
[10/42] MERCEDES-BENZ 1218 R (2001) | código 509048-2...
   ↳ ✅ R$ 100.502,00
[11/42] MERCEDES-BENZ 1318 (2007) | código 509057-1...
   ↳ ✅ R$ 119.690,00
[12/42] MERCEDES-BENZ 1318 (2008) | código 509057-1...
   ↳ ✅ R$ 136.356,00
[13/42] MERCEDES-BENZ 710 (2008) | código 509032-6...
   ↳ ✅ R$ 139.700,00
[14/42] MERCEDES-BENZ 710 (2010) | código 509032-6...
   ↳ ✅ R$ 149.008,00
[15/42] MERCEDES-BEN

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>